# MapReduce in Python: Word Count

This notebook implements the three phases of MapReduce (**Map**, **Shuffle and Sort**, **Reduce**) as plain Python functions, to count the words in the first lines of Dante's *Divina Commedia*.

On a real cluster each phase would run on many machines at the same time. Here everything runs on one machine, so that you can look at the output of every phase.

Run the cells from top to bottom (**Shift + Enter**).

## Loading the Text
The text is downloaded from the GitHub repository of the course, so the notebook works anywhere (Colab, Databricks or your computer). Each line of the text becomes one element of the list `data`.

In [ ]:
import urllib.request

URL = "https://raw.githubusercontent.com/Naviden/Introduction-to-Big-Data-and-NoSQL/main/data/misc/divina_commedia_first_pages.txt"

with urllib.request.urlopen(URL) as response:
    data = response.read().decode("utf-8").splitlines()

print(len(data), "lines")

In [ ]:
# The first 3 lines
data[:3]

## Map Step

The Map step processes the input data to produce a set of intermediate key-value pairs. Here, we split each sentence into words and map each word to a key-value pair `(word, 1)`.

In [ ]:
def map_function(sentences):
    """
    Map function to split sentences into words and map each word to (word, 1)
    """
    mapped = []
    for sentence in sentences:
        words = sentence.split()
        for word in words:
            mapped.append((word.lower(), 1))
    return mapped

# Execute Map Step
mapped = map_function(data)
print("Number of pairs:", len(mapped))
print("First 10 pairs:", mapped[:10])

## Shuffle and Sort Step

This step groups the intermediate key-value pairs by key (word), and sorts the keys. In a real MapReduce system, this is handled by the framework, and it is the step where data travels over the network: all the pairs with the same key are sent to the same machine. Here, we group and sort the pairs ourselves.

In [ ]:
from collections import defaultdict

def shuffle_and_sort(mapped):
    """
    Shuffle and Sort function to group the mapped key-value pairs by key, sorted by key
    """
    grouped = defaultdict(list)
    for key, value in mapped:
        grouped[key].append(value)
    return dict(sorted(grouped.items()))

# Execute Shuffle and Sort Step
grouped = shuffle_and_sort(mapped)
print("Number of distinct words:", len(grouped))
print("First 10 groups:", list(grouped.items())[:10])

## Reduce Step

The Reduce step merges all intermediate values associated with the same key to produce the final result. Here, we sum the values for each key to get the total count of each word.

In [ ]:
def reduce_function(grouped):
    """
    Reduce function to sum the values for each key to get the total count of each word
    """
    reduced = {}
    for key, values in grouped.items():
        reduced[key] = sum(values)
    return reduced

# Execute Reduce Step
reduced = reduce_function(grouped)

## Final Result

The final result is a dictionary where keys are words and values are their respective counts.

In [ ]:
# Display the final result
for word, count in reduced.items():
    print(f'{word}: {count}')

The 10 most frequent words:

In [ ]:
for word, count in sorted(reduced.items(), key=lambda pair: pair[1], reverse=True)[:10]:
    print(f'{word}: {count}')

## Exercises

1. Look at the final result: `che` and `che,` are counted as two different words (and so are `smarrita` and `smarrita.`). Change `map_function` so that it removes punctuation before counting (hint: `word.strip(".,;:!?")`). How many distinct words are there now?
2. Change the program to count **letters** instead of words. What do the Map and Reduce functions emit now?

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2